# Learn Agent Creation with Ollama


We will move step by step:
1. Check that Ollama is reachable
2. List and select a model
3. Send a simple prompt
4. Use chat history
5. Build tools
6. Create a tiny agent loop

> Make sure Ollama is running locally on `http://localhost:11434` before executing the cells.

In [15]:
import json
import requests
from typing import Any, Dict, List

OLLAMA_URL = "http://localhost:11434"
print("Ollama URL:", OLLAMA_URL)

Ollama URL: http://localhost:11434


## Step 1: Check if Ollama is running

The `/api/tags` endpoint lists the available models. If this works, your local Ollama server is up and ready.

In [16]:
response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=20)
print("Status Code: ", response.status_code)
print(response.text[:500])
models = response.json().get("models", [])
for model in models:
    print(model.get("name"))

Status Code:  200
{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-07-26T21:47:59.2597299+05:30","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16","context_length":2048,"embedding_length":768},"capabilities":["embedding"]},{"name":"llama3.2:latest","model":"llama3.2:latest","modif
nomic-embed-text:latest
llama3.2:latest
llama3.2:3b


## Step 2: Choose a model

You can use any local model installed in Ollama. A common example is `llama3.2` or `mistral`. Replace the model name below if needed.

In [17]:
MODEL_NAME = "llama3.2:latest"
available = {a.get("name") for a in requests.get(f"{OLLAMA_URL}/api/tags", timeout=20).json().get("models", [])}
if MODEL_NAME not in available:
    print(f"Warning: {MODEL_NAME} is not available")
else:
    print(f"Selected model: {MODEL_NAME}")

Selected model: llama3.2:latest


## Step 3: Send a simple prompt to Ollama

This is the most basic pattern: send a prompt and get a response.

In [18]:
def generate_text(model_name:str, prompt: str, system_prompt: str = "You are a helpful assistant") -> str:
    payload = {
        "model": model_name,
        "prompt": prompt,
        "system": system_prompt,
        "stream": False
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=60)
    response.raise_for_status()
    return response.json()["response"]

result = generate_text(MODEL_NAME, "Explain in 2 lines what an AI agent is?")
print(result)

An AI agent is a software program that uses artificial intelligence (AI) and machine learning (ML) to make decisions, take actions, and interact with its environment. It is designed to perform specific tasks autonomously, without human intervention, using complex algorithms and data processing.


## Step 4: Use the chat endpoint with conversation history

For an agent, we usually need a message history instead of a single prompt. That is where the `/api/chat` endpoint is useful.

In [19]:
def chat_with_model(model_name:str, messages: List[Dict[str, str]], temperature: float = 0.7) -> str:
    payload = {
        "model": model_name,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature}
    }
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=60)
    response.raise_for_status()
    return response.json()["message"]["content"]

conversation = [
    {"role": "user", "content": "What is a simple defination of an AI Agent?"},
    {"role": "assistant", "content": "An AI agent is a software system that perceives its environment, takes actions, and learns from its experiences to achieve a specific goal or objective. It uses various techniques, such as machine learning and computer vision, to interact with the world and make decisions autonomously."},
    {"role": "user", "content": "Give me an example in everyday life."}
]
reply = chat_with_model(MODEL_NAME, conversation)
print(reply)

A simple example of an AI agent in everyday life is a smart thermostat.

Imagine you have a smart thermostat that can learn your daily schedule and temperature preferences. It can:

* Perceive the current temperature in your home (environment)
* Take action to adjust the temperature based on your schedule and preferences (action)
* Learn from its experiences and adjust its settings to optimize energy efficiency and comfort (learning)

This smart thermostat is an example of an AI agent, as it uses its sensors to perceive its environment, takes actions to adjust the temperature, and learns from its experiences to improve its performance over time.


## Step 5: Create a tool list

Agents are more useful when they can call functions. Here we define a few simple tools: a calculator and a weather lookup stub.

In [20]:
def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtin__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

def get_weather(city: str) -> str:
    weather_map = {
        "delhi": "32 degree and sunny",
        "london": "18 degree and cloudy",
        "new york": "24 degree and mild"
    }
    return weather_map.get(city.lower(), f"No weather data for {city}")

TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather
}
print(TOOLS["calculator"]("12*7"))
print(TOOLS["get_weather"]("london"))

84
18 degree and cloudy


## Step 6: Build a tiny agent loop

The agent will:
1. Accept user input
2. Ask the model to decide whether it needs a tool
3. If required, run the tool
4. Send the tool result back to the model
5. Return the final answer

This is the heart of a simple local agent.

In [21]:
class OllamaAgent:
    def __init__(self, model_name:str, system_prompt:str ="You are a helpful local agent, use tools only when necessary."):
        self.model_model = model_name
        self.system_prompt = system_prompt
        self.history: List[Dict[str, str]] = []

    def chat(self, messages: List[Dict[str, str]]) -> str:
        payload = {
            "model": self.model_model,
            "messages": messages,
            "stream": False,
            "Options": {"temperature": 0.2}
        }
        response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=60)
        response.raise_for_status()
        return response.json()["message"]["content"]

    def run(self, user_input:str) -> str:
        self.history.append({"role": "user", "content": user_input})

        messages = [{"role": "system", "content": self.system_prompt}] + self.history
        reply = self.chat(messages)
        self.history.append({"role": "system", "content": reply})
        return reply

agent = OllamaAgent(MODEL_NAME)
print(agent.run("What is the capital of India?"))
print(agent.run("What we discussed previously?"))

The capital of India is New Delhi.
We didn't discuss anything previously. This is the start of our conversation. You mentioned that the capital of India is New Delhi.


## Step 7: Add a tool-calling agent

This version tells the model: if needed, return a JSON object with a tool name and arguments. Then we execute the function in Python.

In [22]:
class ToolCallingAgent:
    def __init__(self, model_name:str, tools: Dict[str, Any]):
        self.model_name=model_name
        self.tools=tools
        self.histroy: List[Dict[str, str]] = []

    def call_model(self, messages: List[Dict[str, str]]) -> str:
        payload = {
            "model": self.model_name,
            "messages": messages,
            "stream": False 
        }
        response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=60)
        response.raise_for_status()
        return response.json()["message"]["content"]

    def run(self, user_input:str) -> str:
        tool_instruction = (
            "You are a tool-using agent. If the user ask for calculations or weather, respond with JSON in the format:"
            "{\"tool\": \"calculator\", \"arguments\": {\"expression\": \"3 + 4\"}}"
            "or"
            "{\"tool\": \"get_weather\", \"arguments\": {\"city\": \"Delhi\"}}"
            "If no tool is needed, answer normally, Return only valid JSON using a tool."
        )
        messages = [
            {"role": "system", "content": tool_instruction},
            {"role": "user", "content": user_input},
        ]
        model_output = self.call_model(messages)
        print("Model Output: ", model_output)

        try:
            parsed = json.loads(model_output)
            tool_name = parsed.get("tool")
            arguments = parsed.get("arguments", {})
            if tool_name in self.tools:
                result = self.tools[tool_name](**arguments)
                return f"Tool result: {result}"
        except Exception:
            pass

        return model_output

tool_agent = ToolCallingAgent(MODEL_NAME, TOOLS)
print(tool_agent.run("What is 14*5?"))
print(tool_agent.run("what is the weather is London?"))



Model Output:  {"tool": "calculator", "arguments": {"expression": "14*5"}}
Tool result: 70
Model Output:  {"tool": "get_weather", "arguments": {"city": "London"}}
Tool result: 18 degree and cloudy


## Final notes

This notebook demonstrates the fundamental pattern behind local agents with Ollama: prompt, context, tool use, and looped reasoning. Once you understand this pattern, you can expand it into more advanced workflows with memory, structured outputs, and multi-step planning.